In [1]:
# %%
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import re
from datetime import datetime, timedelta

# =========================================
# CONFIG
# =========================================

UNDERLYING_MAP = {
    "PETR": "PETR4",
    "VALE": "VALE3",
    "BBDC": "BBDC4",
    "BBAS": "BBAS3",
    "ABEV": "ABEV3",
    "WEGE": "WEGE3",
    "B3SA": "B3SA3",
    "SUZB": "SUZB3",
}

UNDERLYINGS_BASE = list(UNDERLYING_MAP.keys())

# ticker válido de opção B3
OPTION_REGEX = re.compile(r"^[A-Z]{4}[A-Z][0-9]{2,3}$")


# =========================================
# FUNÇÃO: BAIXAR ÚLTIMO COTAHIST DISPONÍVEL
# =========================================

def download_latest_cotahist(max_lookback_days=10):

    base_url = "https://bvmf.bmfbovespa.com.br/InstDados/SerHist/COTAHIST_D{date}.ZIP"
    today = datetime.today()

    for i in range(max_lookback_days):

        ref = today - timedelta(days=i)
        date_str = ref.strftime("%d%m%Y")
        url = base_url.format(date=date_str)

        try:

            r = requests.get(url, timeout=20)

            if r.status_code != 200:
                continue

            z = zipfile.ZipFile(io.BytesIO(r.content))
            file_name = z.namelist()[0]
            data = z.read(file_name).decode("latin1")
            lines = data.splitlines()

            print("✅ Arquivo encontrado:", url)
            print("Arquivo interno:", file_name)
            print("Total de linhas:", len(lines))

            return ref.date(), lines

        except Exception:
            continue

    raise RuntimeError("❌ Não foi possível baixar um COTAHIST recente.")


# =========================================
# FUNÇÃO: PARSE DO COTAHIST
# =========================================

def parse_cotahist(lines, underlyings_base):

    records = []

    for l in lines:

        if not l.startswith("01"):
            continue

        ticker = l[12:24].strip().upper()

        if not any(ticker.startswith(u) for u in underlyings_base):
            continue

        close = int(l[108:121]) / 100
        volume = int(l[152:170])

        strike_raw = l[188:201]
        expiration_raw = l[202:210]

        strike = np.nan
        expiration = pd.NaT


        # =====================================
        # STRIKE
        # =====================================

        if strike_raw.strip().isdigit():

            strike_val = int(strike_raw)

            if strike_val != 0:

                strike = strike_val / 100

                # REMOVE STRIKES TERMINADOS EM .0
                # elimina PETRD540, PETRD550 etc
                if strike.is_integer():
                    strike = np.nan


        # =====================================
        # EXPIRATION
        # =====================================

        if expiration_raw.strip().isdigit():

            exp_val = int(expiration_raw)

            if exp_val != 0:
                expiration = datetime.strptime(expiration_raw, "%Y%m%d")


        # =====================================
        # FILTRO DE TICKER REAL DE OPÇÃO
        # =====================================

        if pd.notna(strike):

            # elimina séries fantasmas tipo:
            # PETRD540W5
            # PETRD540M
            # PETRD540T
            # etc

            if not OPTION_REGEX.match(ticker):
                continue


        records.append({

            "ticker": ticker,
            "fechamento_d-1": close,
            "volume_d-1": volume,
            "strike": strike,
            "expiration": expiration

        })

    return pd.DataFrame(records)

# =========================================
# EXECUÇÃO
# =========================================

trade_date, lines = download_latest_cotahist()

df_raw = parse_cotahist(lines, UNDERLYINGS_BASE)

print("Data de referência:", trade_date)
print("Qtd registros após filtro base:", len(df_raw))

display(df_raw.head(20))

✅ Arquivo encontrado: https://bvmf.bmfbovespa.com.br/InstDados/SerHist/COTAHIST_D19032026.ZIP
Arquivo interno: COTAHIST_D19032026.TXT
Total de linhas: 19754
Data de referência: 2026-03-19
Qtd registros após filtro base: 3555


,ticker,fechamento_d-1,volume_d-1,strike,expiration
0,VALE3,76.63,27262900,NaN,9999-12-31
1,PETR3,51.57,18102600,NaN,9999-12-31
2,PETR4,46.78,84865000,NaN,9999-12-31
3,SUZB3,51.12,8891500,NaN,9999-12-31
4,BBAS3,23.51,29427500,NaN,9999-12-31
5,WEGE3,46.59,5299300,NaN,9999-12-31
6,BBDC3,16.11,6082900,NaN,9999-12-31
7,BBDC4,18.64,33436700,NaN,9999-12-31
8,ABEV3,14.77,30848100,NaN,9999-12-31
9,B3SA3,16.92,59021100,NaN,9999-12-31


In [2]:
# %%
from datetime import datetime, timedelta

# =========================================
# 3ª SEXTA
# =========================================

def third_friday(year, month):
    d = datetime(year, month, 1)
    days_until_friday = (4 - d.weekday()) % 7
    first_friday = d + timedelta(days=days_until_friday)
    third = first_friday + timedelta(days=14)
    return third.date()


# usa a data do arquivo, não o relógio do PC
now = datetime.combine(trade_date, datetime.min.time())

current_expiry_date = third_friday(now.year, now.month)

if now.month == 12:
    next_month = 1
    next_year = now.year + 1
else:
    next_month = now.month + 1
    next_year = now.year

next_expiry_date = third_friday(next_year, next_month)

print("Vencimento mês atual :", current_expiry_date)
print("Vencimento próximo mês:", next_expiry_date)


# =========================================
# CLASSIFICAÇÃO
# =========================================

def classify_asset(ticker, strike):
    if pd.notna(strike):
        letter = ticker[4]

        if letter in list("ABCDEFGHIJKL"):
            return "CALL"
        elif letter in list("MNOPQRSTUVWX"):
            return "PUT"
        else:
            return "OPTION"

    return "SPOT"


df_raw["type"] = df_raw.apply(
    lambda r: classify_asset(r["ticker"], r["strike"]),
    axis=1
)


# =========================================
# UNDERLYING CORRETO
# =========================================

def infer_underlying(ticker, asset_type):
    base = ticker[:4]

    if asset_type == "SPOT":
        return ticker

    return UNDERLYING_MAP.get(base)


df_raw["underlying"] = df_raw.apply(
    lambda r: infer_underlying(r["ticker"], r["type"]),
    axis=1
)

print(df_raw["type"].value_counts(dropna=False))
display(df_raw.head(20))

Vencimento mês atual : 2026-03-20
Vencimento próximo mês: 2026-04-17
type
CALL    1706
PUT     1402
SPOT     447
Name: count, dtype: int64


,ticker,fechamento_d-1,volume_d-1,strike,expiration,type,underlying
0,VALE3,76.63,27262900,NaN,9999-12-31,SPOT,VALE3
1,PETR3,51.57,18102600,NaN,9999-12-31,SPOT,PETR3
2,PETR4,46.78,84865000,NaN,9999-12-31,SPOT,PETR4
3,SUZB3,51.12,8891500,NaN,9999-12-31,SPOT,SUZB3
4,BBAS3,23.51,29427500,NaN,9999-12-31,SPOT,BBAS3
5,WEGE3,46.59,5299300,NaN,9999-12-31,SPOT,WEGE3
6,BBDC3,16.11,6082900,NaN,9999-12-31,SPOT,BBDC3
7,BBDC4,18.64,33436700,NaN,9999-12-31,SPOT,BBDC4
8,ABEV3,14.77,30848100,NaN,9999-12-31,SPOT,ABEV3
9,B3SA3,16.92,59021100,NaN,9999-12-31,SPOT,B3SA3


In [3]:
# %%
# =========================================
# UNIVERSO IGUAL AO MT5
# =========================================

df_spot = df_raw[df_raw["type"] == "SPOT"].copy()

df_calls = df_raw[
    (df_raw["type"] == "CALL") &
    (df_raw["expiration"].notna()) &
    (df_raw["expiration"].dt.date.isin([current_expiry_date, next_expiry_date]))
].copy()

print("SPOT:", len(df_spot))
print("CALL:", len(df_calls))


# =========================================
# PREÇO DO SPOT NAS CALLs
# =========================================

spot_price_map = dict(zip(df_spot["ticker"], df_spot["fechamento_d-1"]))
df_calls["spot_price"] = df_calls["underlying"].map(spot_price_map)

print("CALLs com spot mapeado:", df_calls["spot_price"].notna().sum())
display(df_calls.head(20))


# =========================================
# CALLs OTM
# =========================================

df_calls_valid = df_calls[
    (df_calls["strike"].notna()) &
    (df_calls["spot_price"].notna()) &
    (df_calls["strike"] > df_calls["spot_price"])
].copy()

print("CALLs OTM:", len(df_calls_valid))

display(
    df_calls_valid[
        ["ticker", "underlying", "spot_price", "strike", "expiration", "fechamento_d-1", "volume_d-1"]
    ].head(20)
)


# =========================================
# GERAR CALL CREDIT SPREADS
# =========================================

spreads = []

group_cols = ["underlying", "expiration"]

for (underlying, expiration), g in df_calls_valid.groupby(group_cols):

    g = g.sort_values("strike")

    for _, sell in g.iterrows():

        for _, buy in g.iterrows():

            if buy["strike"] <= sell["strike"]:
                continue

            credit = sell["fechamento_d-1"] - buy["fechamento_d-1"]

            if credit <= 0:
                continue

            spreads.append({

                "underlying": underlying,
                "expiration": expiration,

                # TICKERS REAIS
                "sell_ticker": sell["ticker"],
                "buy_ticker": buy["ticker"],

                "sell_strike": sell["strike"],
                "buy_strike": buy["strike"],

                "spot_price": sell["spot_price"],

                "sell_premium": sell["fechamento_d-1"],
                "buy_premium": buy["fechamento_d-1"],
                "credit": credit,

                "sell_volume": sell["volume_d-1"],
                "buy_volume": buy["volume_d-1"],
            })

df_call_spreads = pd.DataFrame(spreads)

print("Qtd spreads gerados:", len(df_call_spreads))
display(df_call_spreads.head(20))

SPOT: 447
CALL: 962
CALLs com spot mapeado: 962


,ticker,fechamento_d-1,volume_d-1,strike,expiration,type,underlying,spot_price
24,ABEVC116,3.77,30300,10.94,2026-03-20,CALL,ABEV3,14.77
25,ABEVC118,3.40,5100,11.19,2026-03-20,CALL,ABEV3,14.77
26,ABEVC123,3.03,159500,11.69,2026-03-20,CALL,ABEV3,14.77
27,ABEVC126,2.70,10000,11.94,2026-03-20,CALL,ABEV3,14.77
28,ABEVC128,2.47,98900,12.19,2026-03-20,CALL,ABEV3,14.77
30,ABEVC131,2.28,92300,12.44,2026-03-20,CALL,ABEV3,14.77
31,ABEVC133,2.10,182300,12.69,2026-03-20,CALL,ABEV3,14.77
32,ABEVC136,1.70,14800,12.94,2026-03-20,CALL,ABEV3,14.77
33,ABEVC138,1.39,48200,13.19,2026-03-20,CALL,ABEV3,14.77
34,ABEVC141,1.18,20600,13.44,2026-03-20,CALL,ABEV3,14.77


CALLs OTM: 357


,ticker,underlying,spot_price,strike,expiration,fechamento_d-1,volume_d-1
41,ABEVC156,ABEV3,14.77,14.94,2026-03-20,0.04,1251400
42,ABEVC158,ABEV3,14.77,15.19,2026-03-20,0.01,124600
44,ABEVC161,ABEV3,14.77,15.44,2026-03-20,0.01,23500
45,ABEVC166,ABEV3,14.77,15.94,2026-03-20,0.01,55600
46,ABEVC168,ABEV3,14.77,16.19,2026-03-20,0.01,100
47,ABEVC183,ABEV3,14.77,17.69,2026-03-20,0.01,500
48,ABEVC228,ABEV3,14.77,22.19,2026-03-20,0.01,30000
65,ABEVD149,ABEV3,14.77,14.92,2026-04-17,0.46,148400
66,ABEVD151,ABEV3,14.77,15.17,2026-04-17,0.34,53000
67,ABEVD154,ABEV3,14.77,15.42,2026-04-17,0.23,227800


Qtd spreads gerados: 4714


,underlying,expiration,sell_ticker,buy_ticker,sell_strike,buy_strike,spot_price,sell_premium,buy_premium,credit,sell_volume,buy_volume
0,ABEV3,2026-03-20,ABEVC156,ABEVC158,14.94,15.19,14.77,0.04,0.01,0.03,1251400,124600
1,ABEV3,2026-03-20,ABEVC156,ABEVC161,14.94,15.44,14.77,0.04,0.01,0.03,1251400,23500
2,ABEV3,2026-03-20,ABEVC156,ABEVC166,14.94,15.94,14.77,0.04,0.01,0.03,1251400,55600
3,ABEV3,2026-03-20,ABEVC156,ABEVC168,14.94,16.19,14.77,0.04,0.01,0.03,1251400,100
4,ABEV3,2026-03-20,ABEVC156,ABEVC183,14.94,17.69,14.77,0.04,0.01,0.03,1251400,500
5,ABEV3,2026-03-20,ABEVC156,ABEVC228,14.94,22.19,14.77,0.04,0.01,0.03,1251400,30000
6,ABEV3,2026-04-17,ABEVD149,ABEVD151,14.92,15.17,14.77,0.46,0.34,0.12,148400,53000
7,ABEV3,2026-04-17,ABEVD149,ABEVD154,14.92,15.42,14.77,0.46,0.23,0.23,148400,227800
8,ABEV3,2026-04-17,ABEVD149,ABEVD156,14.92,15.67,14.77,0.46,0.18,0.28,148400,138800
9,ABEV3,2026-04-17,ABEVD149,ABEVD159,14.92,15.92,14.77,0.46,0.12,0.34,148400,700


In [4]:
# %%
# =========================================
# PARÂMETROS
# =========================================

MIN_SELL_VOLUME    = 5000
MIN_BUY_VOLUME     = 1000
MIN_OTM_PCT        = 2.0
MIN_RETURN_PCT     = 30.0
MIN_CREDIT         = 0.5

MIN_SPREAD_WIDTH   = 1.0
MAX_SPREAD_WIDTH   = 2.0


# =========================================
# MÉTRICAS (IGUAL AO MT5)
# =========================================

df = df_call_spreads.copy()

df["spread_width"] = df["buy_strike"] - df["sell_strike"]

df["otm_pct"] = (
    (df["sell_strike"] - df["spot_price"]) / df["spot_price"]
) * 100

df["max_loss"] = df["spread_width"] - df["credit"]

df["return_pct"] = (df["credit"] / df["max_loss"]) * 100


# =========================================
# FILTROS
# =========================================

df_filt = df[
    (df["sell_volume"]   >= MIN_SELL_VOLUME)   &
    (df["buy_volume"]    >= MIN_BUY_VOLUME)    &
    (df["otm_pct"]       >= MIN_OTM_PCT)       &
    (df["return_pct"]    >= MIN_RETURN_PCT)    &
    (df["credit"]        >= MIN_CREDIT)        &
    (df["spread_width"]  >= MIN_SPREAD_WIDTH)  &
    (df["spread_width"]  <= MAX_SPREAD_WIDTH)  &
    (df["max_loss"]      > 0)
].copy()

print("Qtd travas após filtros:", len(df_filt))


# =========================================
# SEPARA VENCIMENTOS
# =========================================

df_filt["exp_date"] = df_filt["expiration"].dt.date

df_current = df_filt[df_filt["exp_date"] == current_expiry_date]
df_next    = df_filt[df_filt["exp_date"] == next_expiry_date]


# =========================================
# EXIBIÇÃO
# =========================================

def show_table(df_input, titulo):
    print("")
    print("================================")
    print(titulo)
    print("================================")

    if df_input.empty:
        print("Nenhuma trava encontrada")
        return

    df_rank = df_input.sort_values(
        ["return_pct", "otm_pct"],
        ascending=[False, False]
    )

    display(df_rank.head(20))


show_table(df_current, "TRAVAS — VENCIMENTO MÊS ATUAL")
show_table(df_next, "TRAVAS — VENCIMENTO PRÓXIMO MÊS")

Qtd travas após filtros: 25

TRAVAS — VENCIMENTO MÊS ATUAL
Nenhuma trava encontrada

TRAVAS — VENCIMENTO PRÓXIMO MÊS


,underlying,expiration,sell_ticker,buy_ticker,sell_strike,buy_strike,spot_price,sell_premium,buy_premium,credit,sell_volume,buy_volume,spread_width,otm_pct,max_loss,return_pct,exp_date
4518,WEGE3,2026-04-17,WEGED488,WEGED498,47.61,48.61,46.59,1.34,0.77,0.57,59900,5700,1.00,2.189311,0.43,132.558140,2026-04-17
4575,WEGE3,2026-04-17,WEGED482,WEGED492,48.36,49.36,46.59,1.07,0.57,0.50,18300,3500,1.00,3.799099,0.50,100.000000,2026-04-17
4558,WEGE3,2026-04-17,WEGED493,WEGED492,48.11,49.36,46.59,1.15,0.57,0.58,128800,3500,1.25,3.262503,0.67,86.567164,2026-04-17
3032,VALE3,2026-04-17,VALED784,VALED799,78.40,79.90,76.63,2.29,1.63,0.66,855200,407200,1.50,2.309800,0.84,78.571429,2026-04-17
4521,WEGE3,2026-04-17,WEGED488,WEGED492,47.61,49.36,46.59,1.34,0.57,0.77,59900,3500,1.75,2.189311,0.98,78.571429,2026-04-17
3033,VALE3,2026-04-17,VALED784,VALED804,78.40,80.40,76.63,2.29,1.47,0.82,855200,729400,2.00,2.309800,1.18,69.491525,2026-04-17
1347,PETR4,2026-04-17,PETRD479,PETRD496,47.90,49.65,46.78,1.78,1.15,0.63,992900,814900,1.75,2.394186,1.12,56.250000,2026-04-17
2351,SUZB3,2026-04-17,SUZBD535,SUZBD555,52.38,54.38,51.12,1.46,0.75,0.71,46100,42300,2.00,2.464789,1.29,55.038760,2026-04-17
3083,VALE3,2026-04-17,VALED789,VALED804,78.90,80.40,76.63,2.00,1.47,0.53,757700,729400,1.50,2.962286,0.97,54.639175,2026-04-17
1387,PETR4,2026-04-17,PETRD481,PETRD496,48.15,49.65,46.78,1.67,1.15,0.52,1773800,814900,1.50,2.928602,0.98,53.061224,2026-04-17


In [5]:
# =========================================
# TELEGRAM
# =========================================

import requests

TELEGRAM_BOT_TOKEN = "8363974523:AAGGmbw7gfIUlCnjkYVSBLKXbiShQhA5yGg"
TELEGRAM_CHAT_ID = "8564196952"


def send_telegram_message(text):

    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"

    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": text,
        "parse_mode": "Markdown"
    }

    requests.post(url, json=payload)

In [6]:
def format_spread_message(row):

    msg = f"""
📊 *Trava de Crédito Detectada*

{row.sell_ticker} / {row.buy_ticker}

Spot: {row.spot_price:.2f}

SELL strike: {row.sell_strike:.2f}
BUY strike:  {row.buy_strike:.2f}

Crédito: {row.credit:.2f}
Risco:   {row.max_loss:.2f}

Width: {row.spread_width:.2f}

OTM: {row.otm_pct:.2f}%
Retorno: {row.return_pct:.2f}%
"""

    return msg

In [7]:
# =========================================
# TELEGRAM ALERT — BLOOMBERG STYLE
# =========================================

df_rank_next = df_filt[
    df_filt["exp_date"] == next_expiry_date
].sort_values(
    ["return_pct", "otm_pct"],
    ascending=[False, False]
)

top_spreads = df_rank_next.head(10)

msg = "📊 *TOP 10 TRAVAS - PRÓX VCTO*\n\n"
msg += "`SELL/BUY        OTM   CR   RET`\n"

for row in top_spreads.itertuples():

    sell = row.sell_ticker
    buy  = row.buy_ticker

    msg += (
        f"`{sell}/{buy[-3:]} "
        f"{row.otm_pct:>5.2f} "
        f"{row.credit:>4.2f} "
        f"{row.return_pct:>4.0f}`\n"
    )

send_telegram_message(msg)